<a href="https://colab.research.google.com/github/TonyQ2k3/pytorch-training/blob/main/mlflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pytorch Day 2
---

In [ ]:
!pip install mlflow

In [22]:
import mlflow
import mlflow.pyfunc
import mlflow.spark
import os
import re

from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, udf, col
from pyspark.ml import PipelineModel

In [26]:
import pandas as pd

data = pd.read_csv('/content/Google Pixel_2025-05-15_00-16-54.csv')

In [27]:
data.head()

,product,text,author,score,created
0,Google Pixel,Pixel 9 Pro reportedly costs Google around 400...,a_Ninja_b0y,1873,2024-11-05
1,Google Pixel,"Sounds like a high price, tbh, I mean that’s o...",air_twee,702,2024-11-05
2,Google Pixel,"If I’m not mistaken, that’s in the same ballpa...",kiwipo17,122,2024-11-05
3,Google Pixel,Remember when Pixels were budget flagship phones?,1stltwill,57,2024-11-05
4,Google Pixel,So 999$ in retail is not that much If you add ...,Azuras33,225,2024-11-05


In [25]:
# Clean tweets and remove unwanted characters
def clean_text(text):
    if text is not None:
        # Remove any URLs and irrelevant characters
        text = re.sub(r'https?://\S+|www\.\S+', '', text)
        text = re.sub(r'[\U0001F600-\U0001F64F]|[\U0001F300-\U0001F5FF]|[\U0001F680-\U0001F6FF]|[\U0001F700-\U0001F77F]|[\U0001F800-\U0001F8FF]|[\U0001F900-\U0001F9FF]|[\U0001FA00-\U0001FAFF]', '', text)
        text = re.sub(r'!\[gif\]', '', text)
        text = re.sub(r'\[deleted\]', '', text)

        # Remove tag-words starting with # or @
        text = re.sub(r'(@|#)\w+', '', text)

        # Convert to lowercase
        text = text.lower()

        # Remove non-alphanumeric characters
        text = re.sub(r'[^a-zA-Z\s]', '', text)

        # Remove extra whitespaces
        text = re.sub(r'\s+', ' ', text).strip()
        return text
    else:
        return ''

In [10]:
def load_model_from_mlflow(model_uri):
    """
    Load a model from MLflow.
    :param model_uri: URI of the model in MLflow.
    :return: Loaded model.
    """
    # Load the model
    model = mlflow.spark.load_model(model_uri)
    return model

In [11]:
mlflow.set_tracking_uri("https://dagshub.com/TranChucThien/kltn-sentiment-monitoring-mlops.mlflow")
spark = SparkSession.builder \
.appName("Load CountVectorizer_Model from MLflow") \
.getOrCreate()

# Load the model from MLflow
model_uri = "models:/CountVectorizer_Model/1"
model = load_model_from_mlflow(model_uri)

2025/05/31 09:19:08 INFO mlflow.spark: URI 'models:/CountVectorizer_Model/1/sparkml' does not point to the current DFS.
2025/05/31 09:19:08 INFO mlflow.spark: File 'models:/CountVectorizer_Model/1/sparkml' not found on DFS. Will attempt to upload the file.


In [28]:
# Tạo DataFrame
df = spark.createDataFrame(data)
cleaned_df = df.withColumn("Text", udf(clean_text)(col("text")))

In [29]:
cleaned_df.printSchema()

root
 |-- product: string (nullable = true)
 |-- Text: string (nullable = true)
 |-- author: string (nullable = true)
 |-- score: long (nullable = true)
 |-- created: string (nullable = true)



In [30]:
# Chạy transform qua model đã load
predictions = model.transform(cleaned_df)

# Hiển thị kết quả
predictions.show(truncate=False)

+------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------+-----+----------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [ ]:
def map_class(value):
  class_index_mapping = { 0: "Negative", 1: "Positive", 2: "Neutral" }
  return class_index_mapping[int(value)]

DataFrame[product: string, Text: string, author: string, score: bigint, created: string, Label: double, words: array<string>, filtered_words: array<string>, features: vector, rawPrediction: vector, probability: vector, prediction: double, class: string]

In [ ]:
predictions = predictions.withColumn("class", udf(map_class)(col("prediction")))

In [ ]:
predictions.select("class", "Text").show(truncate=False)

+--------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|class   |Text                                                                                                                                                                                                                                                                                                                                                                                                                                                                   